# 01 — Exploratory Data Analysis: IEEE-CIS Fraud Detection

**Goal:** understand the data well enough to make three decisions:

1. **How to validate.** Does fraud behavior drift over time? If so, a random split overstates performance.
2. **Which signals exist.** What do amount, product, card, email, device and missingness tell us?
3. **What the model is really predicting.** Are fraud labels attached to *transactions* or to *customers*?
   This decides the feature strategy (entity resolution, per-customer aggregates).

Data: ~590k card-not-present transactions from Vesta Corporation, joined with identity/device data.
Run `make data` first. The raw data is not committed to this repo.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))  # allow running without `pip install -e .`
from fraud_detection import config
from fraud_detection.data import load_raw

pd.set_option("display.max_columns", 50)
plt.rcParams.update({"figure.figsize": (10, 4), "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.3})
FRAUD, LEGIT = "#c0392b", "#7f8c8d"

In [ ]:
df = load_raw()

# Downcast floats to cut memory roughly in half (~430 columns).
float_cols = df.select_dtypes("float64").columns
df[float_cols] = df[float_cols].astype("float32")

T = config.TARGET
DAY = 86_400
df["day"] = (df[config.TIME_COL] // DAY).astype(int)

print(f"{len(df):,} transactions × {df.shape[1]} columns, {df.memory_usage().sum() / 1e9:.2f} GB in memory")
print(f"Fraud rate: {df[T].mean():.2%}  ({df[T].sum():,} frauds)")
print(f"Time span: {df['day'].min()}–{df['day'].max()} days")

## 1. Column groups

Vesta anonymized most features. The column-name prefixes tell us what each group represents:

| Prefix | Meaning (per competition host) |
|---|---|
| `card1–6` | card attributes: issuer, network, type, etc. |
| `addr1–2`, `dist1–2` | billing region / country, distances |
| `P_/R_emaildomain` | purchaser / recipient email domain |
| `C1–14` | counts, e.g. how many addresses are tied to the card |
| `D1–15` | timedeltas, e.g. days since previous transaction |
| `M1–9` | match flags, e.g. name on card vs. address |
| `V1–339` | Vesta-engineered features |
| `id_01–38`, `Device*` | identity: network, browser, device |

In [ ]:
import re

def group_of(col):
    m = re.match(r"^(card|addr|dist|C|D|M|V|id_)\d", col)
    return m.group(1).rstrip("_") if m else col

groups = (
    pd.DataFrame({"col": df.columns, "group": [group_of(c) for c in df.columns],
                  "missing": df.isna().mean().values})
    .groupby("group").agg(n_cols=("col", "size"), avg_missing=("missing", "mean"))
    .query("n_cols > 1").sort_values("n_cols", ascending=False)
)
groups.style.format({"avg_missing": "{:.0%}"})

## 2. Fraud over time: choosing the validation split

The competition's test set comes **after** the training period. A deployed model also scores
future transactions. If the fraud rate or fraud patterns drift, a random split leaks the future
into training and overstates performance.

In [ ]:
daily = df.groupby("day").agg(n=(T, "size"), fraud_rate=(T, "mean"))
weekly = daily["fraud_rate"].rolling(7, center=True).mean()
cut_day = df[config.TIME_COL].quantile(1 - config.VALID_FRACTION) // DAY

fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
ax[0].bar(daily.index, daily["n"], color=LEGIT, width=1)
ax[0].set_ylabel("transactions / day")
ax[1].plot(daily.index, daily["fraud_rate"], color=FRAUD, alpha=0.3, lw=1, label="daily")
ax[1].plot(weekly.index, weekly, color=FRAUD, lw=2, label="7-day avg")
ax[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.1%}"))
ax[1].set_ylabel("fraud rate"); ax[1].set_xlabel("day")
for a in ax:
    a.axvline(cut_day, color="black", ls="--", lw=1)
ax[0].text(cut_day, ax[0].get_ylim()[1] * 0.9, "  validation →", va="top")
ax[1].legend(loc="upper left")
fig.suptitle("Volume and fraud rate over time")
plt.tight_layout(); plt.show()

print(f"Fraud rate: train period {df.loc[df['day'] < cut_day, T].mean():.2%} | "
      f"validation period {df.loc[df['day'] >= cut_day, T].mean():.2%}")

## 3. Transaction amount

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
bins = np.linspace(0, np.log10(df["TransactionAmt"].max() + 1), 60)
for label, color, name in [(0, LEGIT, "legit"), (1, FRAUD, "fraud")]:
    ax[0].hist(np.log10(df.loc[df[T] == label, "TransactionAmt"] + 1), bins=bins,
               density=True, alpha=0.6, color=color, label=name)
ax[0].set_xlabel("log10(amount + 1)"); ax[0].set_ylabel("density"); ax[0].legend()
ax[0].set_title("Amount distribution by class")

# Fraud rate by amount decile
dec = pd.qcut(df["TransactionAmt"], 10, duplicates="drop")
by_dec = df.groupby(dec, observed=True)[T].mean()
ax[1].bar(range(len(by_dec)), by_dec.values, color=FRAUD)
ax[1].set_xticks(range(len(by_dec)), [f"{iv.right:,.0f}" for iv in by_dec.index], rotation=45)
ax[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.1%}"))
ax[1].axhline(df[T].mean(), color="black", ls="--", lw=1)
ax[1].set_xlabel("amount decile (upper bound, $)"); ax[1].set_title("Fraud rate by amount decile")
plt.tight_layout(); plt.show()

The **cents** part of the amount often carries signal. Round amounts, odd cents, and
currency-converted amounts like `.95` vs `.xyz` with 3 decimals can indicate the purchaser's
country or currency.

In [ ]:
cents = (df["TransactionAmt"] % 1).round(3)
has_3dp = ((df["TransactionAmt"] * 1000).round() % 10 != 0)
summary = pd.DataFrame({
    "share": [ (cents == 0).mean(), has_3dp.mean() ],
    "fraud_rate": [ df.loc[cents == 0, T].mean(), df.loc[has_3dp, T].mean() ],
}, index=["whole-dollar amount", "3+ decimal places"])
summary.loc["overall"] = [1.0, df[T].mean()]
summary.style.format("{:.2%}")

## 4. Categorical signals

Each chart shows volume (bars) and fraud rate (dots). A useful category is one with both
**meaningful volume** and a **fraud rate far from the 3.4% average**.

In [ ]:
def cat_plot(col, top=12, ax=None):
    s = df[col].astype("object").fillna("(missing)")
    stats = df.groupby(s)[T].agg(["size", "mean"]).sort_values("size", ascending=False).head(top)
    ax = ax or plt.gca()
    ax.bar(range(len(stats)), stats["size"], color=LEGIT, alpha=0.5)
    ax.set_xticks(range(len(stats)), stats.index, rotation=45, ha="right")
    ax.set_ylabel("transactions")
    ax2 = ax.twinx()
    ax2.plot(range(len(stats)), stats["mean"], "o", color=FRAUD)
    ax2.axhline(df[T].mean(), color=FRAUD, ls="--", lw=1)
    ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
    ax2.set_ylim(0, max(0.05, stats["mean"].max() * 1.15)); ax2.grid(False)
    ax.set_title(col)

cat_cols = [c for c in ["ProductCD", "card4", "card6", "DeviceType", "P_emaildomain", "R_emaildomain"]
            if c in df]
ncols = 2
nrows = int(np.ceil(len(cat_cols) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(13, 4 * nrows))
for c, a in zip(cat_cols, np.ravel(axes)):
    cat_plot(c, ax=a)
for a in np.ravel(axes)[len(cat_cols):]:
    a.set_visible(False)
plt.tight_layout(); plt.show()

## 5. Missingness is a feature

In this dataset whether a field is filled in often depends on the product or channel, for example
whether identity data was collected at all. So "missing" carries information. LightGBM handles
NaN natively, so it can learn these patterns. The table shows the columns where fraud rates differ
most between missing and present.

In [ ]:
miss = df.drop(columns=[T]).isna()
rows = []
for c in miss.columns[miss.any()]:
    m = miss[c]
    if 0.01 < m.mean() < 0.99:
        rows.append((c, m.mean(), df.loc[m, T].mean(), df.loc[~m, T].mean()))
miss_tbl = pd.DataFrame(rows, columns=["column", "missing_share", "fraud_if_missing", "fraud_if_present"])
miss_tbl["ratio"] = miss_tbl["fraud_if_present"] / miss_tbl["fraud_if_missing"].clip(lower=1e-6)
miss_tbl = miss_tbl.sort_values("ratio", key=lambda r: np.abs(np.log(r.clip(1e-6))), ascending=False)
miss_tbl.head(15).style.format({"missing_share": "{:.0%}", "fraud_if_missing": "{:.2%}",
                                "fraud_if_present": "{:.2%}", "ratio": "{:.1f}×"})

In [ ]:
# The V columns are missing in blocks: columns that share the same missing count were probably
# produced by the same upstream process. Useful for dimensionality reduction later.
v_cols = [c for c in df.columns if re.fullmatch(r"V\d+", c)]
if v_cols:
    blocks = df[v_cols].isna().sum().rename("n_missing").reset_index().groupby("n_missing")["index"].apply(list)
    print(f"{len(v_cols)} V columns fall into {len(blocks)} missingness blocks")
    display(blocks.apply(len).rename("n_columns").to_frame().sort_values("n_columns", ascending=False).head(10))

## 6. Who is the fraudster? Entity resolution

Vesta's labeling rule (per the competition host): once a transaction is reported as fraud, **later
transactions tied to the same user account, email or billing address are also labeled fraud.** So
labels behave more like *customer* labels than *transaction* labels.

There's no customer ID in the data, but one can be approximated:
- `card1` + `addr1` + `P_emaildomain` gives a rough card holder.
- `D1` is days since the card's first transaction, so `day − D1` is a fixed "account start day".
  It stays constant across all of one customer's transactions, which makes it a strong part of the key.

If labels really are customer-level, nearly every pseudo-customer should be **all fraud or all
legit.**

In [ ]:
need = ["card1", "addr1", "P_emaildomain", "D1"]
if all(c in df for c in need):
    start_day = (df["day"] - df["D1"]).round()
    parts = [df["card1"], df["addr1"], df["P_emaildomain"], start_day]
    key = parts[0].astype("string")
    for p in parts[1:]:
        key = key + "_" + p.astype("string").fillna("NA")
    # No customer ID without card1 and D1: leave those rows out of the customer analysis.
    df["uid"] = key.where(df["card1"].notna() & df["D1"].notna())
    print(f"{df['uid'].notna().mean():.1%} of transactions get a pseudo-customer ID")

    g = df.groupby("uid")[T].agg(n="size", frauds="sum")
    multi = g[g["n"] > 1]
    mixed = multi[(multi["frauds"] > 0) & (multi["frauds"] < multi["n"])]
    fraud_uids = multi[multi["frauds"] > 0]

    print(f"{df['uid'].nunique():,} pseudo-customers; {len(multi):,} with 2+ transactions")
    if len(fraud_uids):
        print(f"Of the {len(fraud_uids):,} multi-transaction customers with any fraud, "
              f"{1 - len(mixed) / len(fraud_uids):.1%} are labeled fraud on EVERY transaction")
    else:
        print("No repeat customers with fraud found. Check the UID definition.")

    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].hist(np.log10(g["n"]), bins=40, color=LEGIT)
    ax[0].set_xlabel("log10(transactions per customer)"); ax[0].set_title("Transactions per pseudo-customer")
    share = (fraud_uids["frauds"] / fraud_uids["n"])
    ax[1].hist(share, bins=20, color=FRAUD)
    ax[1].set_xlabel("share of customer's transactions labeled fraud")
    ax[1].set_title("Customers with ≥1 fraud: label consistency")
    plt.tight_layout(); plt.show()
else:
    print("Skipping: requires", need)

In [ ]:
# Implication for validation: how many validation-period customers were already seen in training?
if "uid" in df:
    train_uids = set(df.loc[df["day"] < cut_day, "uid"])
    train_uids.discard(np.nan)
    valid = df[(df["day"] >= cut_day) & df["uid"].notna()]
    seen = valid["uid"].isin(train_uids)
    print(f"Validation transactions from customers already seen in training: {seen.mean():.1%}")
    print(f"Fraud rate | seen customers: {valid.loc[seen, T].mean():.2%}   "
          f"new customers: {valid.loc[~seen, T].mean():.2%}")

## 7. Takeaways → modeling decisions

*Fill in with the actual numbers after running. Each takeaway should point to a decision.*

| Finding | Decision |
|---|---|
| Fraud rate and volume shift over time (§2) | Keep the **time-based split**. Random splits would overstate performance. |
| Amount and cents patterns differ for fraud (§3) | Keep `log_amt` and `amt_cents`; add amount relative to the customer's typical amount. |
| Some products, email domains and devices have a much higher fraud rate (§4) | Keep the categorical features; consider frequency encoding for high-cardinality columns. |
| Missingness relates strongly to fraud (§5) | Leave NaNs for LightGBM; add missing-count features; reduce V columns by block. |
| Labels are mostly consistent within a pseudo-customer (§6) | **Next: build the UID and per-customer aggregates.** This is the main lever. Also report performance for seen vs. new customers. |